<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day1/notebooks/2_classic_cta_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 1 — Classic Text Analysis & the Document-Term Matrix

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

In the previous notebook we worked with a tiny toy corpus. Now we move to a **real dataset**
that political scientists have studied for decades: the **U.S. presidential inaugural
addresses**, from Washington (1789) to the present.

By the end of this notebook you will be able to:

- Load and explore a real text corpus with metadata (here: the year of each speech)
- Reapply the preprocessing pipeline to real-world text
- Build a **document-term matrix (DTM)** with scikit-learn
- Inspect and interpret the DTM
- Track how word usage changes **over time**

> **How to use this notebook:** run each cell in order with `Shift + Enter`. Wherever you
> see a **✏️ Exercise**, try it before moving on.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import inaugural, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

# Download the NLTK data we need (safe to run more than once)
nltk.download("inaugural")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

print("Setup complete!")

## 1. Loading a real corpus

NLTK ships with the inaugural address corpus built in. Each document is identified by a
filename like `2009-Obama.txt`, which conveniently encodes both the **year** and the
**president**.


In [ ]:
# The file identifiers
file_ids = inaugural.fileids()

print("Number of addresses:", len(file_ids))
print("First:", file_ids[0])
print("Last: ", file_ids[-1])
print()
print("A few in the middle:")
for fid in file_ids[28:32]:
    print("  ", fid)

Let's pull the **raw text**, the **year**, and the **president** into a tidy
`pandas` DataFrame. This is a very common first step: getting your corpus into a table
where each row is a document.


In [ ]:
records = []
for fid in file_ids:
    year = int(fid[:4])                 # first 4 characters are the year
    president = fid[5:-4]               # between the dash and ".txt"
    text = inaugural.raw(fid)
    records.append({"year": year, "president": president, "text": text})

df = pd.DataFrame(records)
print("DataFrame shape:", df.shape)
df.head()

> **✏️ Exercise 1**
>
> Using the DataFrame `df`, print the **year** and **president** of the earliest and the
> most recent address. *(Hint: `df["year"].min()` and boolean filtering, or `.iloc`.)*


In [ ]:
# Your code here


## 2. A first look at the texts

Before any modeling, always *look* at your data. Let's check how long the speeches are,
and how that has changed over time.


In [ ]:
# Add a simple word-count column (rough: split on whitespace)
df["n_words"] = df["text"].apply(lambda t: len(t.split()))

print(df[["year", "president", "n_words"]].head(10))
print()
print("Shortest speech:")
print(df.loc[df["n_words"].idxmin(), ["year", "president", "n_words"]])
print()
print("Longest speech:")
print(df.loc[df["n_words"].idxmax(), ["year", "president", "n_words"]])

In [ ]:
# Plot speech length over time
plt.figure(figsize=(11, 5))
plt.plot(df["year"], df["n_words"], marker="o", color="#34B233")
plt.title("Length of inaugural addresses over time")
plt.xlabel("Year")
plt.ylabel("Number of words")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **✏️ Exercise 2**
>
> Do longer speeches cluster in a particular era? Look at the plot and write a sentence
> (as a comment or in a markdown cell) describing what you notice. Then find the president
> who gave the longest address.


In [ ]:
# Your code here


## 3. Preprocessing (again!)

Just like last time, we clean the text before analysis. The pipeline is the same idea —
lowercase, tokenize, drop punctuation and stopwords, lemmatize — but now applied to real,
messier text.

We wrap it in a function so we can reuse it on all 60 speeches.


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    """Clean a single document and return a space-joined string of tokens."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]        # drop punctuation/numbers
    tokens = [t for t in tokens if t not in stop_words]  # drop stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]   # lemmatize
    return " ".join(tokens)

# Test on the first 200 characters of Obama's 2009 speech
obama = df.loc[df["president"] == "Obama", "text"].iloc[0]
print("RAW:\n", obama[:200])
print("\nCLEANED:\n", preprocess(obama[:200]))

We return a **space-joined string** rather than a list this time. That is because the
scikit-learn tool we use next expects each document as a string. Let's clean the whole
corpus.


In [ ]:
df["clean"] = df["text"].apply(preprocess)

# Peek at the cleaned version
print(df.loc[0, "clean"][:300], "...")

> **✏️ Exercise 3**
>
> Our `preprocess` function always removes stopwords and always lemmatizes. Add an
> argument `remove_stopwords=True` so you can turn stopword removal on and off, like we
> did in the Day 1 notebook. Test it on Obama's speech with stopwords **kept**.


In [ ]:
# Your code here


## 4. Building the Document-Term Matrix

This is the heart of the session. A **document-term matrix (DTM)** has:

- one **row per document**,
- one **column per term** (word in the vocabulary),
- each **cell** = how many times that term appears in that document.

We *could* build it by hand with `Counter` (like Day 1), but scikit-learn's
`CountVectorizer` does it efficiently and is what you will use in practice.


In [ ]:
# Create the vectorizer.
# min_df=2 keeps only words appearing in at least 2 documents (drops rare noise).
vectorizer = CountVectorizer(min_df=2)

# Fit on our cleaned text and transform into a DTM
dtm = vectorizer.fit_transform(df["clean"])

print("DTM shape (documents x terms):", dtm.shape)
print("That is", dtm.shape[0], "speeches and", dtm.shape[1], "unique terms.")

The DTM is stored as a **sparse matrix** — because most cells are zero (most words do
not appear in most documents), storing only the non-zero entries saves huge amounts of
memory. This is exactly the *sparsity* point from the lecture.


In [ ]:
# How sparse is it?
n_cells = dtm.shape[0] * dtm.shape[1]
n_nonzero = dtm.nnz
print(f"Total cells:      {n_cells:,}")
print(f"Non-zero cells:   {n_nonzero:,}")
print(f"Percentage zeros: {100 * (1 - n_nonzero / n_cells):.1f}%")

To actually *look* at the DTM, we convert it to a DataFrame. **Only do this for small
corpora** — for large ones it would blow up your memory. Here, 60 speeches is fine.


In [ ]:
# Convert to a readable DataFrame
dtm_df = pd.DataFrame(
    dtm.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=df["president"] + " (" + df["year"].astype(str) + ")",
)

print("DTM as a table (first 5 docs, a few columns):")
dtm_df.iloc[:5, :8]

> **✏️ Exercise 4**
>
> Look up how many times the word **"freedom"** appears in each speech. Which president
> used it most? *(Hint: `dtm_df["freedom"]` gives you that column; `.sort_values()` and
> `.idxmax()` are handy.)*


In [ ]:
# Your code here


## 5. What can we do with a DTM?

Once text is a matrix of numbers, lots of analysis becomes simple arithmetic.

**Total word frequencies** = sum each column down all documents.


In [ ]:
# Sum each term's counts across all documents
total_counts = dtm_df.sum(axis=0).sort_values(ascending=False)

print("Top 15 words across all inaugural addresses:")
print(total_counts.head(15))

In [ ]:
# Plot them
top15 = total_counts.head(15)

plt.figure(figsize=(11, 5))
plt.bar(top15.index, top15.values, color="#34B233")
plt.title("Most frequent words across all inaugural addresses")
plt.xlabel("Word")
plt.ylabel("Total frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **✏️ Exercise 5**
>
> The words above are common political vocabulary (`people`, `government`, `nation`...).
> Pick **two** words you find interesting and make a bar chart comparing just those two.


In [ ]:
# Your code here


## 6. Tracking a word over time

Because each document has a **year**, we can trace how the use of a particular word has
risen or fallen across American history. This is where text-as-data gets genuinely
interesting for social science.


In [ ]:
def plot_word_over_time(word):
    """Plot the frequency of a word across the inaugural addresses by year."""
    if word not in dtm_df.columns:
        print(f"'{word}' is not in the vocabulary (maybe too rare, or removed in cleaning).")
        return
    counts = dtm_df[word].values
    plt.figure(figsize=(11, 5))
    plt.plot(df["year"], counts, marker="o", color="#1A1A2E")
    plt.title(f"Use of '{word}' in inaugural addresses over time")
    plt.xlabel("Year")
    plt.ylabel(f"Count of '{word}'")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_word_over_time("freedom")

In [ ]:
plot_word_over_time("war")

> **✏️ Exercise 6**
>
> Use `plot_word_over_time` to explore a few words of your choice. Try `"america"`,
> `"god"`, `"economy"`, or anything you expect to have a historical pattern. Do any of the
> trends surprise you?
>
> ⚠️ Remember words were **lemmatized**, so search for the lemma: `"nation"` not
> `"nations"`, `"citizen"` not `"citizens"`.


In [ ]:
# Your code here


## 7. Bonus: comparing two speeches

A DTM lets us compare documents numerically. A simple approach: which words are most
distinctive to one speech versus another? Here we just look at the raw difference in
counts between two speeches. (In the next session we will see **TF-IDF**, a smarter way
to weight words.)


In [ ]:
def compare_speeches(pres1, pres2, n=10):
    """Show words most distinctive to pres1 vs pres2 (by raw count difference)."""
    row1 = dtm_df.loc[dtm_df.index.str.startswith(pres1)].iloc[0]
    row2 = dtm_df.loc[dtm_df.index.str.startswith(pres2)].iloc[0]
    diff = (row1 - row2).sort_values(ascending=False)
    print(f"Words more used by {pres1}:")
    print(diff.head(n))
    print(f"\nWords more used by {pres2}:")
    print(diff.tail(n)[::-1])

compare_speeches("Lincoln", "Trump")

> **✏️ Exercise 7**
>
> Use `compare_speeches` to compare two presidents of your choice. Remember to use a
> surname that appears in the data (e.g. `"Roosevelt"` matches more than one — try
> `"Obama"`, `"Kennedy"`, `"Reagan"`, `"Bush"`).


In [ ]:
# Your code here


## Wrap-up

In this notebook you have:

- Loaded a **real political science corpus** (inaugural addresses, 1789–present)
- Organized it into a `pandas` DataFrame with **year** and **president** metadata
- Reapplied the **preprocessing pipeline** to real text
- Built a **document-term matrix** with scikit-learn's `CountVectorizer`
- Explored **sparsity**, total frequencies, and change **over time**
- Compared documents numerically

The DTM is the workhorse representation behind almost all classic text analysis. Next, we build on it with **TF-IDF weighting** and start doing dictionary-based
and supervised analysis.

### Optional challenge

The raw counts in the DTM favor **longer speeches** (more words = more of everything). Try
**normalizing** each row so it sums to 1 (i.e. convert counts to *proportions*), then
redo the "freedom over time" plot. Does the picture change?

*(Hint: `dtm_df.div(dtm_df.sum(axis=1), axis=0)` divides each row by its total.)*


In [ ]:
# Optional challenge — your code here
